# Analysis of DRUNet (UNetRes) as an adaptive linear transform

DRUNet operates on its input by applying an adaptive linear transform whose weights depend on the input image and the noise level. We analyse this transform using singular value decomposition, replicating the analysis from `bias_free_denoising/analysis_demo.ipynb` for the DRUNet architecture implemented in `utils/models.py`.

Because DRUNet uses a concatenated noise map as extra input (non-blind mode), the Jacobian is taken only with respect to the noisy image, treating the noise map as a fixed constant.

In [ ]:
import sys
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import torch

# Point to the project root so all existing utils are importable
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.models import UNetRes
from utils.utils import normalize, remove_dataparallel_wrapper, compute_noise_map

%matplotlib widget

In [ ]:
# ── User-configurable parameters ────────────────────────────────────────────

# Path to a DRUNet (UNetRes) checkpoint
MODEL_PATH = '../TRAINING_LOGS/mix_acutance/ckpt.pth'

# Path to a test image (grayscale analysis works on any image)
IMAGE_PATH = '../datasets/test_sets/Kodak24/kodim15.png'

# Patch to analyse: [row_start:row_end, col_start:col_end].
# Keep it small (≤50×50) — Jacobian computation is O(N²).
PATCH_ROW = slice(100, 140)
PATCH_COL = slice(100, 140)

# Noise levels used in the interactive filter visualisation (0–255 scale)
NOISE_LEVELS = [10, 30, 90]

# Noise level used for the SVD analysis (0–255 scale)
NOISE_SIGMA_SVD = 90

# Model settings — must match the checkpoint
BLIND  = False   # True → model takes only the noisy image (in_nc = num_channels)
IN_CH  = 1       # image channels (1 = grayscale, 3 = RGB)

USE_CUDA = torch.cuda.is_available()
device   = torch.device('cuda' if USE_CUDA else 'cpu')
print(f'Using device: {device}')

## Load model

In [ ]:
def load_drunet(model_path, in_ch=1, blind=False, device=torch.device('cpu')):
    """Load a UNetRes checkpoint, handling DataParallel wrappers and ckpt dicts."""
    # Non-blind DRUNet concatenates a noise map channel to the input
    model_in_nc = in_ch if blind else in_ch + 1
    network = UNetRes(in_nc=model_in_nc, out_nc=in_ch).to(device)

    raw = torch.load(model_path, map_location='cpu')
    state_dict = raw.get('state_dict', raw)
    if any(k.startswith('module.') for k in state_dict):
        state_dict = remove_dataparallel_wrapper(state_dict)
    network.load_state_dict(state_dict)
    network.eval()
    print(f'Loaded DRUNet from {model_path}')
    return network


model = load_drunet(MODEL_PATH, in_ch=IN_CH, blind=BLIND, device=device)

## Load image and select patch

We convert to grayscale for the Jacobian analysis so the spatial structure is interpretable. The same approach applies to RGB images, but the Jacobian grows to $(3N)^2$.

In [ ]:
raw_bgr = cv2.imread(IMAGE_PATH)
if raw_bgr is None:
    raise FileNotFoundError(f'Could not read {IMAGE_PATH}')

image_rgb  = normalize(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))  # HxWx3, float32 [0,1]
image_gray = np.mean(image_rgb, axis=2)                            # HxW

clean_patch = image_gray[PATCH_ROW, PATCH_COL]
patch_h, patch_w = clean_patch.shape
print(f'Patch size: {patch_h}×{patch_w}  (N={patch_h*patch_w} pixels)')

fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(clean_patch, cmap='gray', vmin=0, vmax=1)
ax.axis('off')
ax.set_title('Clean patch')
plt.tight_layout()
plt.show()

## Interpretation: nonlinear adaptive filtering

DRUNet can be viewed as applying an adaptive linear transform: each output pixel is a weighted sum of input pixels. The weights (one row of the Jacobian) form the *adaptive filter* for that pixel. Here we visualise these filters interactively for several noise levels.

### Interactive visualisation — click on any noisy image to see its adaptive filter

In [ ]:
def denoise_drunet_grad(image_2d, model, noise_sigma, device, requires_grad=True):
    """Forward pass keeping the computation graph for gradient computation.

    Returns (denoised_tensor 1×1×H×W, input_tensor 1×1×H×W).
    The input tensor has requires_grad=requires_grad.
    """
    H, W = image_2d.shape
    inp = torch.tensor(image_2d.astype('float32'), requires_grad=requires_grad)
    inp = inp.unsqueeze(0).unsqueeze(0).to(device)
    noise_map = torch.full((1, 1, H, W), noise_sigma, device=device)
    model_input = torch.cat([inp, noise_map], dim=1)
    residual = model(model_input)
    denoised = torch.clamp(inp - residual, 0., 1.)
    return denoised, inp


# Freeze model weights so autograd only tracks the input
for param in model.parameters():
    param.requires_grad = False

# Pre-compute noisy inputs and denoised outputs for each noise level
noisy_patches, denoised_tensors, input_tensors = [], [], []
for sigma in NOISE_LEVELS:
    noisy = clean_patch + np.random.normal(0, sigma / 255., clean_patch.shape)
    noisy_patches.append(noisy)
    den, inp_t = denoise_drunet_grad(noisy, model, sigma / 255., device)
    denoised_tensors.append(den)
    input_tensors.append(inp_t)


# ── Build figure ────────────────────────────────────────────────────────────
n_cols = len(NOISE_LEVELS)
fig = plt.figure(figsize=(8, 9))
fig.suptitle('Click on a pixel in any noisy image to see its adaptive filter')

noisy_axes = []
for j, sigma in enumerate(NOISE_LEVELS):
    ax = fig.add_subplot(3, n_cols, j + 1)
    ax.imshow(noisy_patches[j], cmap='gray', vmin=0, vmax=1)
    ax.set_title(rf'$\sigma={sigma}$ / noisy')
    ax.tick_params(bottom=False, left=False, labelleft=False, labelbottom=False)
    noisy_axes.append(ax)

    ax2 = fig.add_subplot(3, n_cols, j + n_cols + 1)
    den_np = denoised_tensors[j][0, 0].detach().cpu().numpy()
    ax2.imshow(den_np, cmap='gray', vmin=0, vmax=1)
    ax2.set_title('denoised')
    ax2.tick_params(bottom=False, left=False, labelleft=False, labelbottom=False)


# ── Click handler ────────────────────────────────────────────────────────────
pixel_markers = []

def on_click(event):
    global pixel_markers
    if event.xdata is None or event.ydata is None:
        return
    px, py = int(event.xdata), int(event.ydata)

    for marker in pixel_markers:
        marker.set_visible(False)
    pixel_markers.clear()

    for j in range(n_cols):
        # Compute adaptive filter: gradient of output pixel w.r.t. input
        adaptive_filter = torch.autograd.grad(
            denoised_tensors[j][0, 0, py, px],
            input_tensors[j],
            retain_graph=True
        )[0][0, 0].detach().cpu().numpy()

        ax = fig.add_subplot(3, n_cols, j + 2 * n_cols + 1)
        limit = max(np.abs(adaptive_filter.min()), np.abs(adaptive_filter.max()))
        ax.imshow(adaptive_filter, cmap='RdGy', vmin=-limit, vmax=limit)
        ax.set_title('adaptive filter\nsum=' + str(np.round(adaptive_filter.sum(), 2)))
        ax.tick_params(bottom=False, left=False, labelleft=False, labelbottom=False)

        rect = patches.Rectangle(
            (px - 0.5, py - 0.5), 1, 1,
            edgecolor='red', facecolor='red', linewidth=1
        )
        noisy_axes[j].add_patch(rect)
        pixel_markers.append(rect)

    fig.canvas.draw_idle()


click_handler_id = fig.canvas.mpl_connect('button_press_event', on_click)
plt.tight_layout()
plt.show()

In [ ]:
# Disconnect the click handler when done with the interactive exploration
fig.canvas.mpl_disconnect(click_handler_id)

## Interpretation: projection onto an adaptive signal subspace

A singular value decomposition of the full Jacobian $A$ reveals:
1. Most singular values are close to zero — the network discards all but a low-dimensional part of its input.
2. The left and right singular vectors of the signal subspace are nearly identical — the network is projecting the noisy input onto a low-dimensional subspace.

### Compute SVD of the Jacobian

We evaluate the Jacobian at a noisy version of the clean patch. **Warning: slow — expect 1–3 minutes for a 40×40 patch.**

In [ ]:
def calc_jacobian_drunet(image_2d, model, noise_sigma, device):
    """Compute the full Jacobian of the DRUNet denoiser at the given image.

    J[i, j] = d(denoised_pixel_i) / d(noisy_pixel_j).
    The noise map is treated as a fixed constant (not differentiated through).

    Returns a (N, N) numpy array where N = H * W.
    """
    H, W = image_2d.shape
    inp = torch.tensor(image_2d.astype('float32'), requires_grad=True)
    inp = inp.unsqueeze(0).unsqueeze(0).to(device)   # 1×1×H×W
    noise_map = torch.full((1, 1, H, W), noise_sigma, device=device)

    residual = model(torch.cat([inp, noise_map], dim=1))
    denoised = inp - residual   # 1×1×H×W

    rows = []
    for i in range(H):
        for j in range(W):
            grad = torch.autograd.grad(
                denoised[0, 0, i, j], inp, retain_graph=True
            )[0]
            rows.append(grad[0, 0].detach().cpu().view(-1))

    return torch.stack(rows).numpy()   # (N, N)


# Add noise at the chosen SVD sigma level
noisy_patch_svd = clean_patch + np.random.normal(0, NOISE_SIGMA_SVD / 255., clean_patch.shape)

print(f'Computing Jacobian for sigma={NOISE_SIGMA_SVD}, patch {patch_h}×{patch_w} …')
t0 = time.time()
A = calc_jacobian_drunet(noisy_patch_svd, model, NOISE_SIGMA_SVD / 255., device)
print(f'Done in {time.time() - t0:.1f}s  |  Jacobian shape: {A.shape}')

In [ ]:
U, S, Vt = np.linalg.svd(A)
print(f'Top-5 singular values: {np.round(S[:5], 3)}')
print(f'Fraction of energy in top-10%: {S[:len(S)//10].sum()/S.sum():.1%}')

### Singular value spectrum

Most singular values are near zero, confirming that the network discards all but a small-dimensional subspace of its input.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(S, '-o', alpha=0.8, markersize=2, linewidth=1)
ax.set_xlabel('Axis number')
ax.set_ylabel('Singular value')
ax.set_title(f'Singular values of DRUNet Jacobian  (σ={NOISE_SIGMA_SVD})')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Singular vectors with the largest singular values

These are the image features *preserved* by the denoiser.

In [ ]:
large_indices = [0, 1, 2, 3]   # indices of singular vectors to show

limit = max(np.abs(U[:, large_indices]).max(), np.abs(Vt[large_indices, :]).max())
fig, axs = plt.subplots(1, len(large_indices), figsize=(8, 3))
for k, idx in enumerate(large_indices):
    u = U[:, idx]
    v = Vt[idx, :]
    axs[k].imshow(u.reshape(patch_h, patch_w), cmap='gray',
                  vmin=-limit / 2, vmax=limit / 2)
    axs[k].set_title(f'left s.v. {idx}', fontsize=10)
    axs[k].tick_params(bottom=False, left=False, labelleft=False, labelbottom=False)
    axs[k].set_xlabel(
        rf'$v^Tu$ = {np.dot(u, v):.2f}' + '\n' + rf'$s$ = {S[idx]:.2f}',
        fontsize=9
    )
fig.suptitle('Singular vectors with the largest singular values (preserved features)')
plt.tight_layout()
plt.show()

### Singular vectors with the smallest singular values

These are the image features *discarded* by the denoiser (i.e., treated as noise).

In [ ]:
N = patch_h * patch_w
small_indices = [N - 4, N - 3, N - 2, N - 1]

limit = max(np.abs(U[:, small_indices]).max(), np.abs(Vt[small_indices, :]).max())
fig, axs = plt.subplots(1, len(small_indices), figsize=(8, 3))
for k, idx in enumerate(small_indices):
    u = U[:, idx]
    v = Vt[idx, :]
    axs[k].imshow(u.reshape(patch_h, patch_w), cmap='gray',
                  vmin=-limit / 2, vmax=limit / 2)
    axs[k].set_title(f'left s.v. {idx}', fontsize=10)
    axs[k].tick_params(bottom=False, left=False, labelleft=False, labelbottom=False)
    axs[k].set_xlabel(
        rf'$v^Tu$ = {np.dot(u, v):.2f}' + '\n' + rf'$s$ = {S[idx]:.4f}',
        fontsize=9
    )
fig.suptitle('Singular vectors with the smallest singular values (discarded features)')
plt.tight_layout()
plt.show()

### Alignment of left and right singular vectors

For the axes that are *preserved* (large singular values), the left singular vector $u$ and right singular vector $v$ should be nearly identical ($v^Tu \approx 1$). This confirms that the network acts as a projection operator on its signal subspace.

In [ ]:
# Estimate the signal subspace dimension from the energy in squared singular values
subspace_dim = int(np.sum(S ** 2))   # heuristic from the reference notebook
print(f'Estimated signal subspace dimension: {subspace_dim} / {N}')

uv_dots = [abs(np.dot(U[:, j], Vt[j, :])) for j in range(N)]

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(S, uv_dots, '.', color='lightgray', markersize=2, label='discarded axes')
ax.plot(S[:subspace_dim], uv_dots[:subspace_dim], '.', color='green',
        markersize=3, alpha=1, label='preserved axes')
ax.set_xlabel('Singular value')
ax.set_ylabel(r'$|v^T u|$')
ax.set_title('Alignment of left and right singular vectors')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()